# Giai đoạn 1b: Tiền xử lý Dữ liệu BDD100k (Domain Split)
Trích xuất BDD100k thành hai miền `bdd_day` và `bdd_night`, xử lý tọa độ bounding boxes sang định dạng YOLO, và áp dụng cơ chế chống tràn RAM thông qua `ijson` stream parsing.

In [ ]:
import os
import json
import shutil
import ijson
from tqdm import tqdm

# --- CẤU HÌNH ĐƯỜNG DẪN ---
BASE_DIR = 'd:/DAT301m/proposal'
RAW_DATA_DIR = os.path.join(BASE_DIR, 'data/raw/bdd100k_night')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data/processed')

# Đường dẫn file JSON nguyên bản
JSON_TRAIN_PATH = os.path.join(RAW_DATA_DIR, 'labels/bdd100k_labels_images_train.json')
JSON_VAL_PATH = os.path.join(RAW_DATA_DIR, 'labels/bdd100k_labels_images_val.json')

# Thư mục ảnh nguyên bản
IMG_TRAIN_DIR = os.path.join(RAW_DATA_DIR, 'images/train')
IMG_VAL_DIR = os.path.join(RAW_DATA_DIR, 'images/val')

# --- CẤU HÌNH LỚP (CLASSES) ---
BDD_CLASSES = {
    "pedestrian": 0, "rider": 1, "car": 2, "truck": 3, "bus": 4, 
    "train": 5, "motorcycle": 6, "bicycle": 7, "traffic light": 8, "traffic sign": 9
}

IMG_WIDTH = 1280.0
IMG_HEIGHT = 720.0\n

In [ ]:
def create_dirs(base_path):
    for split in ['train', 'val']:
        os.makedirs(os.path.join(base_path, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(base_path, split, 'labels'), exist_ok=True)

def safe_link_or_copy(src, dst):
    if os.path.exists(dst): return
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def build_image_map(base_dir):
    img_map = {}
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.endswith(('.jpg', '.png')):
                img_map[f] = os.path.join(root, f)
    return img_map\n

In [ ]:
def process_split(json_path, raw_img_dir, split_name):
    print(f"\n--- Bắt đầu xử lý tập: {split_name.upper()} ---")
    if not os.path.exists(json_path):
        print(f"Không tìm thấy file: {json_path}")
        return

    day_dir = os.path.join(PROCESSED_DIR, 'bdd_day')
    night_dir = os.path.join(PROCESSED_DIR, 'bdd_night')
    create_dirs(day_dir)
    create_dirs(night_dir)

    print("Đang lập bản đồ đường dẫn ảnh (Scanning directories)...")
    img_map = build_image_map(raw_img_dir)

    stats = {'daytime': 0, 'night': 0, 'dawn/dusk': 0, 'missing_img': 0, 'empty_labels': 0}
    total_est = 70000 if split_name == 'train' else 10000
    
    with open(json_path, 'r', encoding='utf-8') as f:
        items = ijson.items(f, 'item')
        for item in tqdm(items, total=total_est, desc=f"Đọc {split_name}"):
            img_name = item.get('name', '')
            timeofday = item.get('attributes', {}).get('timeofday', '')

            if timeofday == 'dawn/dusk':
                stats['dawn/dusk'] += 1
                continue
            
            if timeofday == 'daytime':
                target_base = day_dir
                stats['daytime'] += 1
            elif timeofday == 'night':
                target_base = night_dir
                stats['night'] += 1
            else:
                continue

            src_img_path = img_map.get(img_name)
            if not src_img_path:
                stats['missing_img'] += 1
                continue

            yolo_labels = []
            for lbl in item.get('labels', []):
                category = lbl.get('category', '')
                if category not in BDD_CLASSES: continue
                box2d = lbl.get('box2d', {})
                if not box2d: continue
                
                x1, y1 = float(box2d['x1']), float(box2d['y1'])
                x2, y2 = float(box2d['x2']), float(box2d['y2'])
                
                x1, x2 = max(0.0, min(x1, IMG_WIDTH)), max(0.0, min(x2, IMG_WIDTH))
                y1, y2 = max(0.0, min(y1, IMG_HEIGHT)), max(0.0, min(y2, IMG_HEIGHT))

                w, h = (x2 - x1), (y2 - y1)
                x_center, y_center = (x1 + w / 2) / IMG_WIDTH, (y1 + h / 2) / IMG_HEIGHT
                w, h = w / IMG_WIDTH, h / IMG_HEIGHT

                yolo_labels.append(f"{BDD_CLASSES[category]} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

            if len(yolo_labels) == 0: stats['empty_labels'] += 1
            
            dst_img_path = os.path.join(target_base, split_name, 'images', img_name)
            dst_lbl_path = os.path.join(target_base, split_name, 'labels', img_name.replace('.jpg', '.txt'))
            
            with open(dst_lbl_path, 'w', encoding='utf-8') as txt_f:
                txt_f.write('\n'.join(yolo_labels))
                
            safe_link_or_copy(src_img_path, dst_img_path)

    print(f"\nThống kê {split_name}:")
    for k, v in stats.items(): print(f"  - {k}: {v}")\n

In [ ]:
print("=== BẮT ĐẦU QUÁ TRÌNH TIỀN XỬ LÝ BDD100k ===")
process_split(JSON_TRAIN_PATH, IMG_TRAIN_DIR, 'train')
process_split(JSON_VAL_PATH, IMG_VAL_DIR, 'val')
print("\n=== HOÀN TẤT ===")\n